In [4]:
import pickle
import folium
from folium.plugins import HeatMap
import pandas as pd
from pathlib import Path

In [6]:
with open("lfbo/tracks.pkl", "rb") as f:
    tracks = pickle.load(f)

print(f"Loaded {len(tracks)} tracks")

Loaded 1332 tracks


In [ ]:
all_points = pd.concat(
    [t.points[["lat", "lon", "baroaltitude"]].assign(
        icao24=t.icao24,
        category=(
            "arriving"  if t.is_arriving  else
            "departing" if t.is_departing else
            "transiting"
        )
    ) for t in tracks],
    ignore_index=True
).dropna(subset=["lat", "lon"])

print(f"Total points: {len(all_points)}")
print(f"Category breakdown:")
print(all_points["category"].value_counts())

Total points: 113469
Category breakdown:
category
transiting    86334
arriving      17514
departing      9621
Name: count, dtype: int64


In [12]:

LFBO_LAT, LFBO_LON = 43.6293, 1.3673
m = folium.Map(location=[LFBO_LAT, LFBO_LON], zoom_start=12, zoom_control=False, scrollWheelZoom=False, doubleClickZoom=False)

heat_data = all_points[["lat", "lon"]].values.tolist()
HeatMap(heat_data, radius=8, blur=10, min_opacity=0.3).add_to(m)

folium.Marker([LFBO_LAT, LFBO_LON], popup="LFBO Toulouse-Blagnac", icon=folium.Icon(color="red", icon="plane")).add_to(m)

m.save("figures/heatmap_all.html")
print("Saved heatmap_all.html")

Saved heatmap_all.html


In [16]:
Path("figures/additional_heatmaps").mkdir(parents=True, exist_ok=True)

for category in ["arriving", "departing", "transiting"]:
    m = folium.Map(location=[LFBO_LAT, LFBO_LON], zoom_start=12, zoom_control=False, scrollWheelZoom=False, doubleClickZoom=False)
    
    subset = all_points[all_points["category"] == category]
    heat_data = subset[["lat", "lon"]].values.tolist()
    
    HeatMap(
        heat_data,
        radius=8,
        blur=10,
        min_opacity=0.3
    ).add_to(m)
    
    folium.Marker([LFBO_LAT, LFBO_LON], popup="LFBO Toulouse-Blagnac", icon=folium.Icon(color="red", icon="plane")).add_to(m)
    
    m.save(f"figures/additional_heatmaps/heatmap_{category}.html")
    print(f"Saved heatmap_{category}.html")

Saved heatmap_arriving.html
Saved heatmap_departing.html
Saved heatmap_transiting.html


In [17]:
m = folium.Map(location=[LFBO_LAT, LFBO_LON], zoom_start=12, zoom_control=False, scrollWheelZoom=False, doubleClickZoom=False)

categories = {
    "Arriving":   ("arriving",   [1, 0, 0]),  # red
    "Departing":  ("departing",  [0, 1, 0]),  # green
    "Transiting": ("transiting", [0, 0, 1]),  # blue
}

for label, (cat, gradient_color) in categories.items():
    fg = folium.FeatureGroup(name=label, show=True)
    
    subset = all_points[all_points["category"] == cat]
    heat_data = subset[["lat", "lon"]].values.tolist()
    
    r, g, b = gradient_color
    HeatMap(heat_data, radius=8, blur=10, min_opacity=0.3,
        gradient={
            0.2: f"rgb({int(r*100)},0,{int(b*100)})",
            0.5: f"rgb({int(r*200)},{int(g*100)},{int(b*200)})",
            1.0: f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"
        }).add_to(fg)
    fg.add_to(m)

folium.Marker([LFBO_LAT, LFBO_LON], popup="LFBO Toulouse-Blagnac", icon=folium.Icon(color="red", icon="plane")).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("figures/heatmap_layered.html")
print("Saved heatmap_layered.html")

Saved heatmap_layered.html
